In [ ]:
# Install required libraries
!pip install tensorflow pandas numpy nltk seaborn

# === Imports ===
import pandas as pd
import numpy as np
import re
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D,
    Concatenate, Input, LayerNormalization, MultiHeadAttention, Add
)
from tensorflow.keras.callbacks import EarlyStopping

# === Download NLTK stopwords ===
nltk.download("stopwords")
STOPWORDS = set(stopwords.words("english"))

# === Load Dataset ===
data_path = "/content/SemEval2016-Task6-raw-annotations-stance.csv"
df = pd.read_csv(data_path)

# === Clean & Map Labels ===
valid_stances = {"FAVOR": 0, "AGAINST": 1, "NONE": 2}
df = df[df["Stance"].isin(valid_stances.keys())]
df.dropna(subset=["Tweet"], inplace=True)
df["label"] = df["Stance"].map(valid_stances).astype(int)

# === Preprocess Text ===
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = " ".join([word for word in text.split() if word not in STOPWORDS])
    return text

df["cleaned_tweet"] = df["Tweet"].apply(preprocess_text)

# === Tokenization ===
vocab_size = 20000
max_length = 100
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df["cleaned_tweet"])
sequences = tokenizer.texts_to_sequences(df["cleaned_tweet"])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding="post", truncating="post")

# === Train-Test Split ===
X = padded_sequences
y = df["label"].values
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# === Load GloVe Embeddings ===
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
glove_zip_path = "/content/glove.6B.zip"
glove_dir = "/content/glove.6B/"

if not os.path.exists(glove_zip_path):
    !wget $glove_url -O $glove_zip_path
if not os.path.exists(glove_dir):
    os.makedirs(glove_dir)
    with zipfile.ZipFile(glove_zip_path, "r") as z:
        z.extractall(glove_dir)

glove_path = os.path.join(glove_dir, "glove.6B.100d.txt")
embedding_dim = 100
embedding_index = {}
with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype="float32")
        embedding_index[word] = coefs

# === Embedding Matrix ===
word_index = tokenizer.word_index
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if i < vocab_size:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

# === Model Definition ===
def build_model():
    inputs = Input(shape=(max_length,))
    x = Embedding(input_dim=vocab_size, output_dim=embedding_dim,
                  weights=[embedding_matrix], input_length=max_length,
                  trainable=True)(inputs)

    # CNN Branch
    cnn = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
    cnn = GlobalMaxPooling1D()(cnn)

    # BiLSTM + MultiHeadAttention Branch
    bilstm = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3))(x)
    attention_out = MultiHeadAttention(num_heads=4, key_dim=64)(bilstm, bilstm)
    attention_out = Add()([bilstm, attention_out])
    attention_out = LayerNormalization(epsilon=1e-6)(attention_out)
    attention_out = GlobalMaxPooling1D()(attention_out)

    # Merge
    merged = Concatenate()([cnn, attention_out])
    dense = Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.01))(merged)
    dropout = Dropout(0.5)(dense)
    outputs = Dense(3, activation="softmax")(dropout)

    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# === Build and Train ===
model = build_model()
model.summary()
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=20, batch_size=32, callbacks=[early_stopping], verbose=1)

# === Evaluation ===
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=-1)
print(classification_report(y_test, y_pred, target_names=["FAVOR", "AGAINST", "NONE"]))

# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["FAVOR", "AGAINST", "NONE"],
            yticklabels=["FAVOR", "AGAINST", "NONE"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === F1 Score ===
f1 = f1_score(y_test, y_pred, average="weighted")
print(f"F1 Score (Weighted): {f1:.4f}")


